In [13]:
import json
import numpy as np
from scipy.stats import mannwhitneyu

def run_unpaired_statistics(baseline_json_path, proposed_json_path, title="Statistical Significance", n_bootstraps=1000):
    """
    Computes statistical significance using independent arrays (Mann-Whitney U).
    """

    def format_p(p):
        return f"{p:.2e}" if p > 0 else "< 1.00e-300 (Underflow limit)"

    with open(baseline_json_path, 'r', encoding='utf-8') as f:
        data_base = json.load(f)

    with open(proposed_json_path, 'r', encoding='utf-8') as f:
        data_prop = json.load(f)

    # ==========================================
    # 1. INDEPENDENT ERROR EXTRACTION
    # ==========================================
    err_base_abs, err_base_sq = [], []
    for sem in sorted(data_base['dynamic_k'].keys(), key=int):
        yt = np.array(data_base['dynamic_k'][sem]['y_true'], dtype=float)
        yp = np.array(data_base['dynamic_k'][sem]['y_pred'], dtype=float)
        mask = ~np.isnan(yt) & ~np.isnan(yp)
        err_base_abs.extend(np.abs(yt[mask] - yp[mask]))
        err_base_sq.extend((yt[mask] - yp[mask])**2)

    err_prop_abs, err_prop_sq = [], []
    for sem in sorted(data_prop['dynamic_k'].keys(), key=int):
        yt = np.array(data_prop['dynamic_k'][sem]['y_true'], dtype=float)
        yp = np.array(data_prop['dynamic_k'][sem]['y_pred'], dtype=float)
        mask = ~np.isnan(yt) & ~np.isnan(yp)
        err_prop_abs.extend(np.abs(yt[mask] - yp[mask]))
        err_prop_sq.extend((yt[mask] - yp[mask])**2)

    err_base_abs = np.array(err_base_abs)
    err_base_sq = np.array(err_base_sq)
    err_prop_abs = np.array(err_prop_abs)
    err_prop_sq = np.array(err_prop_sq)

    # ==========================================
    # TEST 1 & 2: Mann-Whitney U Test (One-Sided)
    # Testing if Baseline errors are statistically GREATER than Proposed errors
    # ==========================================
    _, mae_p = mannwhitneyu(err_base_abs, err_prop_abs, alternative='greater')
    _, mse_p = mannwhitneyu(err_base_sq, err_prop_sq, alternative='greater')

    # ==========================================
    # TEST 3: HEM (Independent Bootstrap)
    # ==========================================
    np.random.seed(42)
    n_base = len(err_base_sq)
    n_prop = len(err_prop_sq)
    count_prop_worse_or_equal = 0

    for _ in range(n_bootstraps):
        # Sample independently
        idx_base = np.random.randint(0, n_base, n_base)
        idx_prop = np.random.randint(0, n_prop, n_prop)

        # Calculate Baseline HEM
        b_rmse_base = np.sqrt(np.mean(err_base_sq[idx_base]))
        b_mae_base = np.mean(err_base_abs[idx_base])
        b_hem_base = 2 * (b_rmse_base * b_mae_base) / (b_rmse_base + b_mae_base)

        # Calculate Proposed HEM
        b_rmse_prop = np.sqrt(np.mean(err_prop_sq[idx_prop]))
        b_mae_prop = np.mean(err_prop_abs[idx_prop])
        b_hem_prop = 2 * (b_rmse_prop * b_mae_prop) / (b_rmse_prop + b_mae_prop)

        # If PROPOSED is worse (higher error) than BASELINE
        if b_hem_prop >= b_hem_base:
            count_prop_worse_or_equal += 1

    p_val_hem = count_prop_worse_or_equal / n_bootstraps

    # ==========================================
    # PRINT RESULTS
    # ==========================================
    print("======================================================")
    print(title)
    print("======================================================")
    print(f"Total Instances (Baseline): {n_base:,}")
    print(f"Total Instances (Proposed): {n_prop:,}")
    print("-" * 54)
    print(f"1. MAE (Mann-Whitney U on Absolute Errors)")
    print(f"   p-value: {format_p(mae_p)}")
    print(f"2. RMSE (Mann-Whitney U on Squared Errors)")
    print(f"   p-value: {format_p(mse_p)}")
    print(f"3. HEM (Bootstrap Hypothesis Test, iterations={n_bootstraps})")
    if p_val_hem == 0:
        print(f"   p-value: < {1/n_bootstraps:.2e}")
    else:
        print(f"   p-value: {format_p(p_val_hem)}")
    print("======================================================")

In [14]:
run_comprehensive_statistics(
        baseline_json_path='..\\results\\adaptive_regression_results_course_based_KMeans.k1.ALL.NarrowFit.RMSE.json',
        proposed_json_path='..\\results\\adaptive_regression_results_course_based_KMeans.c300.ALL.NarrowFit.RMSE.json',
        title = "STATISTICAL SIGNIFICANCE (Non-Clustered (k = 1) vs. Clustered Approach (t = 300))")

STATISTICAL SIGNIFICANCE (Non-Clustered (k = 1) vs. Clustered Approach (t = 300))
Total Paired Instances (N): 46,827
------------------------------------------------------
1. MAE (Wilcoxon on Absolute Errors)
   p-value: 7.87e-40
2. RMSE (Wilcoxon on Squared Errors)
   p-value: 9.02e-155
3. HEM (Bootstrap Hypothesis Test, iterations=1000)
   p-value: < 1.00e-03


In [16]:
run_unpaired_statistics(
        baseline_json_path='..\\results\\adaptive_regression_results_student_based_KMeans.c300.NarrowFit.RMSE.json', # HIGHER ERROR (Student)
        proposed_json_path='..\\results\\adaptive_regression_results_course_based_KMeans.c300.NarrowFit.RMSE.json', # LOWER ERROR (Course)
        title="STATISTICAL SIGNIFICANCE (Course-based vs. Student-based)"
    )

STATISTICAL SIGNIFICANCE (Course-based vs. Student-based)
Total Instances (Baseline): 46,827
Total Instances (Proposed): 46,827
------------------------------------------------------
1. MAE (Mann-Whitney U on Absolute Errors)
   p-value: < 1.00e-300 (Underflow limit)
2. RMSE (Mann-Whitney U on Squared Errors)
   p-value: < 1.00e-300 (Underflow limit)
3. HEM (Bootstrap Hypothesis Test, iterations=1000)
   p-value: < 1.00e-03


In [18]:
import json
import numpy as np
from scipy.stats import norm

def compute_anhui_significance(proposed_json_path, zhang_rmse, zhang_mae, n_bootstraps=1000):
    print("======================================================")
    print("STATISTICAL SIGNIFICANCE TESTING (Proposed vs. Zhang et al.)")
    print("======================================================")

    # 1. Load Proposed Row-Level Predicted Errors
    with open(proposed_json_path, 'r', encoding='utf-8') as f:
        proposed_data = json.load(f)

    sems = sorted(proposed_data['dynamic_k'].keys(), key=int)
    prop_abs_errors = []
    prop_sq_errors = []

    for sem in sems:
        sem_data = proposed_data['dynamic_k'][sem]
        yt = np.array(sem_data['y_true'], dtype=float)
        yp = np.array(sem_data['y_pred'], dtype=float)

        mask = ~np.isnan(yt) & ~np.isnan(yp) & ~np.isinf(yt) & ~np.isinf(yp)
        prop_abs_errors.extend(np.abs(yt[mask] - yp[mask]))
        prop_sq_errors.extend((yt[mask] - yp[mask])**2)

    prop_abs_errors = np.array(prop_abs_errors)
    prop_sq_errors = np.array(prop_sq_errors)
    n_samples = len(prop_abs_errors)

    print(f"Total Shared Evaluation Instances (N): {n_samples:,}")
    print("-" * 54)

    # 2. Extract Baselines for Both Frameworks
    mu_mae_prop = np.mean(prop_abs_errors)
    var_mae_prop = np.var(prop_abs_errors)
    mu_mse_prop = np.mean(prop_sq_errors)
    var_mse_prop = np.var(prop_sq_errors)

    mu_mae_zhang = zhang_mae
    mu_mse_zhang = zhang_rmse ** 2
    zhang_hem_target = (2 * zhang_rmse * zhang_mae) / (zhang_rmse + zhang_mae)

    # 3. Helper function to translate natural log survival to base-10 scientific notation
    def log_to_scientific_str(log_p):
        log10_val = log_p / np.log(10)
        exponent = int(np.floor(log10_val))
        mantissa = 10 ** (log10_val - exponent)
        return f"{mantissa:.2f}e{exponent}"

    # ==================================================
    # TEST 1: MAE Significance (Asymptotic Z-Test)
    # ==================================================
    se_mae = np.sqrt((var_mae_prop / n_samples) + (var_mae_prop / n_samples))
    z_mae = (mu_mae_zhang - mu_mae_prop) / se_mae
    p_mae_string = log_to_scientific_str(norm.logsf(z_mae))

    print(f"1. MAE (Absolute Errors)")
    print(f"   Proposed Mean: {mu_mae_prop:.4f} | Zhang et al. Mean: {mu_mae_zhang:.4f}")
    print(f"   p-value: {p_mae_string}")

    # ==================================================
    # TEST 2: RMSE Significance (Asymptotic Z-Test on Squares)
    # ==================================================
    se_mse = np.sqrt((var_mse_prop / n_samples) + (var_mse_prop / n_samples))
    z_mse = (mu_mse_zhang - mu_mse_prop) / se_mse
    p_mse_string = log_to_scientific_str(norm.logsf(z_mse))

    print(f"2. RMSE (Squared Error Variance)")
    print(f"   Proposed RMSE: {np.sqrt(mu_mse_prop):.4f} | Zhang et al. RMSE: {zhang_rmse:.4f}")
    print(f"   p-value: {p_mse_string}")

    # ==================================================
    # TEST 3: HEM Significance (1,000-Iteration Independent Bootstrap)
    # ==================================================
    np.random.seed(42)
    count_prop_worse_or_equal = 0

    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n_samples, n_samples)
        b_rmse = np.sqrt(np.mean(prop_sq_errors[indices]))
        b_mae = np.mean(prop_abs_errors[indices])
        b_hem = (2 * b_rmse * b_mae) / (b_rmse + b_mae)

        if b_hem >= zhang_hem_target:
            count_prop_worse_or_equal += 1

    p_val_hem = count_prop_worse_or_equal / n_bootstraps

    print(f"3. HEM (Bootstrap Hypothesis Test, iterations={n_bootstraps})")
    print(f"   Proposed HEM: {(2*np.sqrt(mu_mse_prop)*mu_mae_prop)/(np.sqrt(mu_mse_prop)+mu_mae_prop):.4f} | Zhang et al. HEM: {zhang_hem_target:.4f}")
    if p_val_hem == 0:
        print(f"   p-value: < {1/n_bootstraps:.2e}")
    else:
        print(f"   p-value: {p_val_hem:.2e}")
    print("======================================================")

# Execute
if __name__ == "__main__":
    compute_anhui_significance(
        proposed_json_path='..\\results\\adaptive_regression_results_course_based_KMeans.c300.ANHUI.RMSE.json',
        zhang_rmse=1.0032,
        zhang_mae=0.7494,
        n_bootstraps=1000
    )

STATISTICAL SIGNIFICANCE TESTING (Proposed vs. Zhang et al.)
Total Shared Evaluation Instances (N): 16,044
------------------------------------------------------
1. MAE (Absolute Errors)
   Proposed Mean: 0.6909 | Zhang et al. Mean: 0.7494
   p-value: 4.08e-28
2. RMSE (Squared Error Variance)
   Proposed RMSE: 0.8409 | Zhang et al. RMSE: 1.0032
   p-value: 1.38e-207
3. HEM (Bootstrap Hypothesis Test, iterations=1000)
   Proposed HEM: 0.7586 | Zhang et al. HEM: 0.8579
   p-value: < 1.00e-03
